# Definitions

# Getting the Dataset
We use fiftyone to get the COCO validation dataset, though validation can be changed to get other pieces of the COCO dataset.
We filter the dataset to only those images tagged with persons or domestic animals in them.

## Download Dataset

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
from datasets import load_dataset
from keras import utils

ds = load_dataset('logasja/adversarial_examples_fdf', 'originals', split="test")

In [ ]:
# import tensorflow as tf
# print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


I0000 00:00:1740267262.686175    6203 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740267262.710378    6203 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740267262.710437    6203 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.


# Fawkes

## Attack

In [2]:
from multiprocess import set_start_method
set_start_method("spawn")

In [3]:
hash_map = {
    "extractor_2": "ce703d481db2b83513bbdafa27434703",
    "extractor_0": "94854151fd9077997d69ceda107f9c6b",
}
for key, value in hash_map.items():
    utils.get_file(
        fname="{}.h5".format(key),
        origin="http://mirror.cs.uchicago.edu/fawkes/files/{}.h5".format(key),
        md5_hash=value,
        cache_subdir="model",
    )

In [4]:
def fawkes_transform(x, level):
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
    from fawkes.protection import Fawkes
    from fawkes.utils import Faces, reverse_process_cloaked
    from fawkes.differentiator import FawkesMaskGeneration
    from keras import utils
    import numpy as np
    from tqdm import tqdm
    import os

    IMG_SIZE = 112
    PREPROCESS = "raw"

    sd=1e7
    format="png"
    separate_target=True
    debug=False
    maximize=True
    save_last_on_failed=True

    def generate_cloak_images(protector, image_X, target_emb=None):
        cloaked_image_X = protector.compute(image_X, target_emb)
        return cloaked_image_X


    def preproc(img):
        img = img.convert("RGB")
        img = utils.img_to_array(img)
        return img

    if level == "low":
        fwks = Fawkes("extractor_2", 1, mode="low")
    elif level == "mid":
        fwks = Fawkes("extractor_2", 1, mode="mid")
    elif level == "high":
        fwks = Fawkes("extractor_2", 1, mode="high")

    imgs = x["image"]

    n = len(imgs)

    for i, pimg in tqdm(enumerate(imgs), total=n, leave=False):
        if (os.path.exists("./filtered/" + level + "/" + x["path"][i])):
            continue
        img = preproc(pimg)

        try:
            current_param = "-".join(
                [
                    str(x)
                    for x in [
                        fwks.th,
                        sd,
                        fwks.lr,
                        fwks.max_step,
                        -1,
                        format,
                        separate_target,
                        debug,
                    ]
                ]
            )
            faces = Faces(["./Current Face"], [img], fwks.aligner, verbose=0, no_align=False)
            original_images = faces.cropped_faces

            if len(original_images) == 0:
                raise Exception("No face detected. ")
            original_images = np.array(original_images)

            if current_param != fwks.protector_param:
                fwks.protector_param = current_param
                if fwks.protector is not None:
                    del fwks.protector
                batch_size = len(original_images)
                fwks.protector = FawkesMaskGeneration(
                    fwks.feature_extractors_ls,
                    batch_size=batch_size,
                    mimic_img=True,
                    intensity_range=PREPROCESS,
                    initial_const=sd,
                    learning_rate=fwks.lr,
                    max_iterations=fwks.max_step,
                    l_threshold=fwks.th,
                    verbose=0,
                    maximize=maximize,
                    keep_final=False,
                    image_shape=(IMG_SIZE, IMG_SIZE, 3),
                    loss_method="features",
                    tanh_process=True,
                    save_last_on_failed=save_last_on_failed,
                )
            protected_images = generate_cloak_images(fwks.protector, original_images)
            faces.cloaked_cropped_faces = protected_images

            final_images, _ = faces.merge_faces(
                reverse_process_cloaked(protected_images, preprocess=PREPROCESS),
                reverse_process_cloaked(original_images, preprocess=PREPROCESS),
            )

            utils.array_to_img(final_images[-1].astype(np.uint8)).save("./filtered/" + level + "/" + x["path"][i])
        except Exception:
            pimg.save("./filtered/" + level + "/failed_" + x["path"][i])


In [ ]:
# ds = ds.map(fawkes_transform, batched=True, batch_size=100, num_proc=5, fn_kwargs={"level":"mid"})

Map (num_proc=5):   0%|          | 0/6531 [00:00<?, ? examples/s]

Process SpawnPoolWorker-11:
Process SpawnPoolWorker-8:
Process SpawnPoolWorker-9:
Process SpawnPoolWorker-7:
Process SpawnPoolWorker-10:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/meeko/miniconda3/envs/fawkes/lib/python3.12/site-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/home/meeko/miniconda3/envs/fawkes/lib/python3.12/site-packages/multiprocess/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/meeko/miniconda3/envs/fawkes/lib/python3.12/site-packages/multiprocess/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/home/meeko/miniconda3/envs/fawkes/lib/python3.12/site-packages/datasets/utils/py_utils.py", line 678, in _write_generator_to_queue
    for i, result in enumerate(func(**kwargs)):
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/meeko/miniconda

TimeoutError: 

In [7]:
ds = ds.map(lambda x: x["image"].save("./original/" + x["path"]))

Map:   0%|          | 0/6531 [00:00<?, ? examples/s]

In [15]:
!TF_FORCE_GPU_ALLOW_GROWTH=true TF_CPP_MIN_LOG_LEVEL=3 fawkes -d ./original --mode mid --batch-size 4 --format png

I0000 00:00:1740331518.389046   33842 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740331518.412590   33842 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740331518.412696   33842 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740331518.416547   33842 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740331518.416638   33842 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [ ]:
import os
import numpy as np
from PIL import Image
def files_transform(x):
    img = x["image"]

    h, w, c = np.array(img).shape

    if (os.path.exists("./filtered/" + x["path"])):
        return {"image": Image.open("./filtered/" + x["path"]), "path": x["path"]}
    else:
        return {"image": None, "path": x["path"]}
        # return {"image": Image.fromarray(np.zeros((h,w)), mode="RGB"), "path": x["path"]}

In [ ]:
ds = ds.map(files_transform, num_proc=12)

In [ ]:
ds[6414]["image"]

In [ ]:
ds[500]["image"]

In [ ]:
ds.push_to_hub("logasja/adversarial_examples_fdf", config_name="lwokey", private=True)